# Exploratory Data Analysis on the Insurance Dataset

## About the Dataset

This dataset contains information on the relationship between personal attributes (age, gender, BMI, family size, smoking habits), geographic factors, and their impact on medical insurance charges.

*   **Source:** Arun Jangir, and willian oliveira. (2023). Healthcare Insurance [Data set]. Kaggle. https://doi.org/10.34740/KAGGLE/DSV/6678394

**Features:**

*   **Age:** The insured person's age.
*   **Sex:** Gender (male or female) of the insured.
*   **BMI (Body Mass Index):** A measure of body fat based on height and weight.
*   **Children:** The number of dependents covered.
*   **Smoker:** Whether the insured is a smoker (yes or no).
*   **Region:** The geographic area of coverage.
*   **Charges:** Target variable. The medical insurance costs incurred by the insured person.

-----

## 1\. Setup and Data Preparation

First, let's load our libraries and prepare the **`df_initial`** dataset.

-----

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting styles
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

-----

In [ ]:
import os

data_file = '/content/insurance_south.csv'
if not os.path.exists(data_file):
    print(f"Please upload the file...")
    data_file = None
else:
    print(f"'{data_file}' already exists. Skipping upload.")

if data_file and os.path.exists(data_file):
    df_initial = pd.read_csv(data_file)
    print(f"\nSuccessfully loaded the data.")
    print(f"Total records in the dataset: {len(df_initial)}")
    print(f"Regions present: {df_initial['region'].unique()}")
else:
    print("Dataset not loaded. Cannot proceed.")

-----

## 2\. High-Level Data Overview

Let's get a first look at our data's structure, types, and basic statistics.

-----

In [ ]:
# 1. Data Types and Non-Null Counts
print("--- Data Info ---")
df_initial.info()

# 2. Check for Duplicates
duplicates = df_initial.duplicated().sum()
print(f"\n--- Duplicates ---")
print(f"Found {duplicates} duplicate row(s).")
# For this analysis, we'll keep them, but in a real project, we'd investigate.

# 3. Summary Statistics for Numerical Features
print("\n--- Numerical Statistics ---")
print(df_initial.describe())

# 4. Summary for Categorical Features
print("\n--- Categorical Statistics ---")
print(df_initial.describe(include=['object']))

**Questions:**

* How many records are in our 'initial' dataset after filtering for southern regions?
* Are there any missing (null) values in the dataset?
* What is the mean and standard deviation of the `charges`? What does this suggest about the distribution of charges?
* How does the maximum `charges` compare to the 75th percentile? What does this tell us about the skewness of the data?
* How is the data distributed between the `southeast` and `southwest` regions in our dataset?

-----

## 3\. Univariate Analysis (Analyzing Single Variables)

Let's look at each feature individually, starting with our target variable, `charges`.

### 3.1 Target Variable: `charges`

-----

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
sns.histplot(df_initial['charges'], kde=True, ax=axes[0], bins=40)
axes[0].set_title('Distribution of Medical Charges')
axes[0].set_xlabel('Charges ($)')

# Boxplot
sns.boxplot(x=df_initial['charges'], ax=axes[1])
axes[1].set_title('Boxplot of Medical Charges')
axes[1].set_xlabel('Charges ($)')

plt.suptitle("Target Variable: charges (Highly Right-Skewed)")
plt.show()

# Log-transform for a clearer view
plt.figure(figsize=(8, 5))
log_charges = np.log1p(df_initial['charges'])
sns.histplot(log_charges, kde=True, bins=40)
plt.title('Log-Transformed Distribution of Charges')
plt.xlabel('log(Charges)')
plt.show()

**Questions:**

* Describe the distribution of `charges` based on the histogram and boxplot. Is it symmetrical or skewed? If skewed, in which direction?
* Why might a log-transformation of `charges` be useful for building a linear model? How does the distribution change after the log-transformation?
* The log-transformed distribution appears bimodal. What might this suggest about the underlying groups of people in the data?

### 3.2 Numerical Features: `age`, `bmi`, `children`

-----

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.histplot(df_initial['age'], kde=True, ax=axes[0], bins=20)
axes[0].set_title('Distribution of Age')

sns.histplot(df_initial['bmi'], kde=True, ax=axes[1], bins=20)
axes[1].set_title('Distribution of BMI')

sns.countplot(x='children', data=df_initial, ax=axes[2])
axes[2].set_title('Count of Children (Dependents)')

plt.suptitle("Distribution of Numerical Features")
plt.show()

**Questions Based on Numerical Feature Analysis (`age`, `bmi`, `children`):**

* Describe the distribution of `age`. Are there any age groups that are more represented than others?
* What does the distribution of `bmi` look like? What is the approximate center of this distribution?
* How is the number of `children` distributed in this dataset? Which number of children is most common?

### 3.3 Categorical Features: `sex`, `smoker`, `region`

-----

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.countplot(x='sex', data=df_initial, ax=axes[0])
axes[0].set_title('Gender Distribution')

sns.countplot(x='smoker', data=df_initial, ax=axes[1])
axes[1].set_title('Smoker vs. Non-Smoker')

sns.countplot(x='region', data=df_initial, ax=axes[2])
axes[2].set_title('Region Distribution (In our data)')

plt.suptitle("Distribution of Categorical Features")
plt.show()

# Print exact counts
print(df_initial['smoker'].value_counts())

**Questions:**

* Is the dataset balanced in terms of `sex`?
* What is the ratio of `smokers` to `non-smokers` in this dataset? Is this ratio balanced? Why might this imbalance be important for modeling?
* Based on the data we have loaded, how are the records distributed between the `southeast` and `southwest` regions?

-----

## 4\. Bivariate Analysis (Analyzing Feature vs. Target)

How does each feature relate to the `charges`? This is key for feature selection.

### 4.1 Categorical Features vs. `charges`

-----

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Smoker vs Charges
sns.boxplot(x='smoker', y='charges', data=df_initial, ax=axes[0])
axes[0].set_title('Charges by Smoker Status')

# Sex vs Charges
sns.boxplot(x='sex', y='charges', data=df_initial, ax=axes[1])
axes[1].set_title('Charges by Sex')

# Region vs Charges
sns.boxplot(x='region', y='charges', data=df_initial, ax=axes[2])
axes[2].set_title('Charges by Region (South only)')

plt.suptitle("Categorical Features vs. Medical Charges")
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

**Questions:**

* How does `smoker` status relate to the distribution of `charges`? Which group (`smokers` or `non-smokers`) tends to have higher charges? Why is this insight critical?
* Is there a significant difference in `charges` between males and females based on the boxplot?
* How do `charges` compare between the `southeast` and `southwest` regions in this dataset? What potential issue might this raise?

### 4.2 Numerical Features vs. `charges`

-----

In [ ]:
# We'll use a pairplot, but only for numerical features vs. the target
sns.pairplot(df_initial,
            x_vars=['age', 'bmi', 'children'],
            y_vars=['charges'],
            height=5, aspect=0.8, kind='reg')
plt.suptitle("Numerical Features vs. Medical Charges", y=1.02)
plt.show()

**Questions:**

* Describe the relationship between `age` and `charges` based on the scatter plot. Is there a positive or negative correlation? What might explain the appearance of distinct "lines" of data points?
* What does the relationship between `bmi` and `charges` look like? Is the correlation as strong as the correlation between `age` and `charges`?
* How does the number of `children` appear to relate to `charges`? Is this a strong relationship?

-----

## 5\. Multivariate Analysis (Finding Feature Interactions)

The most powerful insights often come from combining features. Let's test our hypothesis that `smoker` status changes everything.

### 5.1 The "Aha\!" Moment: `bmi` vs. `charges`, split by `smoker`

-----

In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(x='bmi', y='charges', data=df_initial, hue='smoker', alpha=0.7, s=80)
plt.title('Charges vs. BMI (Segmented by Smoker Status)')
plt.xlabel('Body Mass Index (BMI)')
plt.ylabel('Medical Charges ($)')
plt.legend(title='Smoker')
plt.show()

**Questions:**

* How does the relationship between `bmi` and `charges` differ for `non-smokers` compared to `smokers`?
* What does this visualization reveal about the interaction between `bmi` and `smoker` status?
* Why is understanding this feature interaction crucial for building an accurate model?

### 5.2 Correlation Heatmap

Let's quantify the relationships. We'll need to encode our categorical variables into numbers first.

-----

In [ ]:
# Create a copy for the heatmap
df_heatmap = df_initial.copy()

# Manually encode binary features
df_heatmap['smoker_n'] = df_heatmap['smoker'].map({'yes': 1, 'no': 0})
df_heatmap['sex_n'] = df_heatmap['sex'].map({'male': 1, 'female': 0})

# Get numerical features plus our ew encoded ones
numerical_cols = ['age', 'bmi', 'children', 'smoker_n', 'sex_n', 'charges']

# Calculate correlation matrix
corr_matrix = df_heatmap[numerical_cols].corr()

# Plot the heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap (Southern Regions Data)')
plt.show()

**Questions:**

* Which variable has the highest positive correlation with `charges`? Does this confirm our previous observations?
* What is the correlation coefficient between `age` and `charges`? How does it compare to the correlation between `smoker` and `charges`?
* The correlation between `bmi` and `charges` is relatively low (0.19) in the heatmap. Why is this number potentially misleading when considering the previous scatter plot?
* What are the approximate correlation coefficients for `sex` and `children` with `charges`? What does this suggest about their predictive power in a simple linear model?

-----

## 6\. MLOps Project: Key Questions to Consider

This EDA raises critical questions for our MLOps project.

### Data Governance & Provenance

  * **Source:** Where did this data *actually* come from? Is it from a single insurance provider? Is it synthetic? (The Kaggle dataset description is vague).
  * **PII:** The data claims to be anonymous, but `age`, `sex`, `bmi`, and `region` could *potentially* be combined to re-identify individuals (a "mosaic attack"). Are we compliant with data privacy laws like HIPAA?
  * **Access:** Who should have access to this training data? Who can see the production predictions? How do we audit this access?

### Responsible AI (Fairness & Bias)

  * **Inherent Bias:** Our *entire* training set is from the US South (`southwest`, `southeast`). How can we trust a model trained *only* on this data to be fair or accurate for people in the `northeast` or `northwest`?
  * **Protected Attributes:** `age` and `sex` are protected classes. We must evaluate our model for fairness. Does it have significantly different error rates (e.g., higher RMSE) for males vs. females? Or for older vs. younger people?
  * **Regional Bias:** We saw `southeast` has higher charges than `southwest`. If our model learns "people from `southeast` are more expensive," is that a valid statistical finding (e.g., due to a higher cost of living) or is it harmful discrimination (e.g., penalizing a group)? This is a *critical* ethical question.
  * **Transparency:** A person receiving a $40,000 premium quote has a right to an explanation. Can our model provide one? (This points to using SHAP, LIME, or simpler, more interpretable models like linear regression).

### Model Decay & MLOps Strategy

  * **Data Drift:** We *know* we will deploy this model to `northwest` and `northeast`. We *must* assume the data distributions will be different. This is **Covariate Shift**.
      * What is the prevalence of `smokers` in the new regions?
      * What is the average `bmi` or `age` in the new regions?
      * What is the cost of living (and thus baseline `charges`) in these new regions?
  * **Concept Drift:** What if the *relationships* change? What if a new state law in the `northeast` caps costs for smokers, fundamentally breaking our model's most important `smoker * charges` relationship? How would we detect this?
  * **Monitoring Strategy:**
    1.  **What metrics will we monitor in production?** R², RMSE, and MAE are obvious.
    2.  **What is our "trigger"?** What threshold of performance decay (e.g., "R² drops by 15%") will trigger an alert for retraining?
    3.  **How will we monitor data drift?** We should track the statistical distributions (mean, median, std) of all input features (`age`, `bmi`, `smoker` rates) in the new regions and compare them to our southern training data.
  * **Retraining Strategy:** When we get new data from the "real-world" deployment, what is our plan?
    1.  **Just retrain:** Combine all data (south + north) and retrain one "global" model?
    2.  **Separate models:** Build and maintain *four* different models, one for each region? This might be more accurate but is 4x the MLOps work.
    3.  **Fine-tuning:** Use the new northern data to "fine-tune" the existing model?
  * **Feedback Loop:** How quickly can we get the *actual* `charges` (ground truth) for the predictions our model makes? Is it real-time (unlikely), daily, or monthly? This feedback loop delay is a major challenge in production MLOps.